In [1]:
!pip install --upgrade unsloth

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os
p = "/content/drive/MyDrive/NLP/lora_training_data_v2_raft.jsonl"
print("파일 있음?", os.path.exists(p))
print("NLP 폴더 목록:", os.listdir("/content/drive/MyDrive/NLP"))

파일 있음? True
NLP 폴더 목록: ['Chapter 1 - Introduction to Language Models.ipynb', 'lora_training_data_gemini_pro_en.jsonl', 'LOL_Agent_Execution.ipynb', 'lora_training_data_v2_raft.jsonl', 'train_LORA.ipynb']


In [4]:
# ① 설치  ② 마운트 먼저 (런타임 재시작됐으니 다시)
# !pip install --upgrade unsloth
# from google.colab import drive; drive.mount('/content/drive')

from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments, TrainerCallback

DATA_FILE_PATH = "/content/drive/MyDrive/NLP/lora_training_data_v2_raft.jsonl"
SAVE_PATH      = "/content/drive/MyDrive/mistral_lol_lora_v2"

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/mistral-7b-v0.3-bnb-4bit",
    max_seq_length=max_seq_length, dtype=None, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=16, lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=3407,
)

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""
EOS = tokenizer.eos_token
def fmt(ex):
    return {"text":[alpaca_prompt.format(i,inp,o)+EOS
                    for i,inp,o in zip(ex["instruction"],ex["input"],ex["output"])]}
dataset = load_dataset("json", data_files=DATA_FILE_PATH, split="train").map(fmt, batched=True)

# 🔒 100스텝마다 어댑터를 Drive에 직접 저장 (Trainer 체크포인트 X = pickle 버그 회피 + 끊겨도 안전)
class DriveSave(TrainerCallback):
    def on_step_end(self, args, state, control, **kw):
        if state.global_step > 0 and state.global_step % 100 == 0:
            model.save_pretrained(SAVE_PATH); tokenizer.save_pretrained(SAVE_PATH)
            print(f"💾 step {state.global_step} -> Drive 저장됨")

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=dataset,
    dataset_text_field="text", max_seq_length=max_seq_length, dataset_num_proc=2,
    callbacks=[DriveSave()],
    args=TrainingArguments(
        per_device_train_batch_size=2, gradient_accumulation_steps=4, warmup_steps=5,
        num_train_epochs=3, learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10, optim="adamw_8bit", weight_decay=0.01,
        lr_scheduler_type="linear", seed=3407, output_dir="outputs",
        save_strategy="no",          # ← 버그 나는 자동저장 끔
    ),
)
trainer.train()
model.save_pretrained(SAVE_PATH); tokenizer.save_pretrained(SAVE_PATH)   # 최종 저장
print("✅ done ->", SAVE_PATH)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.7: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

...

Step,Training Loss
10,1.005141
20,0.544799
30,0.460330
40,0.391547
50,0.330448
60,0.279153
70,0.198740
80,0.205454
90,0.181615
100,0.160026


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/mistral_lol_lora_v2/tokenizer_config.json.


tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/mistral_lol_lora_v2.


💾 step 100 -> Drive 저장됨
💾 step 200 -> Drive 저장됨
💾 step 300 -> Drive 저장됨
💾 step 400 -> Drive 저장됨
✅ done -> /content/drive/MyDrive/mistral_lol_lora_v2
